In [1]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

## Load Model with 4-bit Quantization

In [2]:
!pip install -q git+https://github.com/huggingface/transformers \
                bitsandbytes \
                accelerate \
                qwen-vl-utils

import torch
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✅ Model loaded successfully!")

# ✅ Correct ways to check device and memory
print(f"   Model dtype:  {next(model.parameters()).dtype}")
print(f"   Model device: {next(model.parameters()).device}")
print(f"   VRAM used:    {torch.cuda.memory_allocated() / 1e9:.2f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✅ Model loaded successfully!
   Model dtype:  torch.bfloat16
   Model device: cuda:0
   VRAM used:    6.40 GB


## Run Inference

In [3]:
import os
import json
import time
import csv
from PIL import Image
import torch
from tqdm import tqdm
from pathlib import Path

TEXT_PROMPT = """
  Analyze the image for VPR spatial weighting in {city}.

  Identify 3-8 key regions and assign weights based on uniqueness for localization in {city}:
  - 1.6–2.0: Spatially unique
  - 1.0–1.5: Moderately distinctive
  - 0.3–0.9: Common
  - 0.0–0.2: Non-localizable (sky, vegetation, people, vehicles)

  Skip regions near weight 1.0. Use coordinates (0.0, 0.0) top-left to (1.0, 1.0) bottom-right.

  [
      {
          "center": [x_coord, y_coord],
          "weight": weight_value,
          "reasoning": "brief_description"
      }
  ]
"""

def process_images(test_folder, text_prompt, model=model, processor=processor):
    test_folder   = Path(test_folder)
    output_folder = test_folder / "att_axis_minimal@qwen3-vl-8b"
    output_folder.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(
        p for p in test_folder.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    timing_records = []

    for image_path in tqdm(image_paths, desc="Processing images"):
        json_path = output_folder / f"{image_path.stem}.json"

        # ⏭️ Skip already processed
        if json_path.exists():
            print(f"⏭️  Skipping: {image_path.name}")
            continue

        # Preprocess
        image = Image.open(image_path).convert("RGB")
        image.thumbnail((512, 512), Image.LANCZOS)

        # Build chat input
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": text_prompt},
            ]
        }]

        text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

        # Inference + timing
        start = time.time()
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=512)
        elapsed = time.time() - start

        output_text = processor.batch_decode(
            output_ids[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )[0]

        # Parse & save JSON
        try:
            result = json.loads(output_text)
        except json.JSONDecodeError:
            result = output_text or None

        json_path.write_text(
            json.dumps({"filename": image_path.name, "result": result}, indent=2)
        )

        timing_records.append({"filename": image_path.name, "processing_time": round(elapsed, 3)})
        print(f"✅ {image_path.name} → {elapsed:.2f}s")

    # Save CSV
    csv_path = output_folder / "timing.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "processing_time"])
        writer.writeheader()
        writer.writerows(timing_records)

    print(f"\n📁 Results saved to: {output_folder}")
    print(f"⏱️  Timing saved to:  {csv_path}")

In [4]:
# --- Run ---
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")  # get from https://huggingface.co/settings/tokens

from google.colab import drive
drive.mount('/content/drive')
# test_folder = "/content/drive/MyDrive/LLM_VPR/sf_flood"
# city = "San Francisco"

test_folder = "/content/drive/MyDrive/LLM_VPR/hk_flood"
city = "Hong Kong"

formatted_prompt = TEXT_PROMPT.replace("{city}", city)
# print(formatted_prompt)

process_images(test_folder, formatted_prompt)

Mounted at /content/drive


Processing images:   1%|          | 1/100 [01:00<1:40:33, 60.94s/it]

✅ Cadogan St%22.282403_114.126075_1.png → 59.96s


Processing images:   2%|▏         | 2/100 [02:01<1:38:52, 60.53s/it]

✅ Center St 25%22.286920_114.142285_1.jpg → 58.83s


...

✅ South Ln%22.285386_114.135559_9.png → 48.77s


Processing images:  99%|█████████▉| 99/100 [1:33:33<00:49, 49.76s/it]

✅ Tak Fung St 18%22.302828_114.191616_1.png → 47.97s


Processing images: 100%|██████████| 100/100 [1:34:28<00:00, 56.69s/it]

✅ Tak Fung St 18%22.302828_114.191616_2.png → 55.62s

📁 Results saved to: /content/drive/MyDrive/LLM_VPR/hk_flood/att_axis_minimal@qwen3-vl-8b
⏱️  Timing saved to:  /content/drive/MyDrive/LLM_VPR/hk_flood/att_axis_minimal@qwen3-vl-8b/timing.csv
